# SHAP Values Deep Dive

## Understanding and Interpreting SHAP Explanations

This notebook provides an in-depth exploration of SHAP (SHapley Additive exPlanations) values and their interpretation.

### Contents
1. What are SHAP Values?
2. Types of SHAP Explainers
3. Local vs Global Explanations
4. Visualization Techniques
5. Advanced Interpretation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
import warnings
warnings.filterwarnings('ignore')

# Add project path
import sys
sys.path.insert(0, '..')

print("SHAP version:", shap.__version__)

## 1. What are SHAP Values?

SHAP values are based on Shapley values from game theory. They explain the prediction of an instance by computing the contribution of each feature to the prediction.

**Key Properties:**
- **Local accuracy**: Sum of SHAP values = prediction - expected value
- **Consistency**: If a feature contributes more, its SHAP value increases
- **Missingness**: Features with no contribution have SHAP = 0

In [ ]:
# Load sample data
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
import xgboost as xgb

# Load data
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train model
model = xgb.XGBClassifier(n_estimators=100, max_depth=5, random_state=42, use_label_encoder=False, eval_metric='logloss')
model.fit(X_train, y_train)

print(f"Model accuracy: {model.score(X_test, y_test):.4f}")

## 2. Types of SHAP Explainers

In [ ]:
# TreeExplainer - Fast for tree-based models
tree_explainer = shap.TreeExplainer(model)
tree_shap_values = tree_explainer.shap_values(X_test[:50])

print("TreeExplainer SHAP values shape:", np.array(tree_shap_values).shape)

In [ ]:
# Explanation object (newer SHAP API)
explanation = tree_explainer(X_test[:50])

print("Explanation object:")
print(f"  - Values shape: {explanation.values.shape}")
print(f"  - Base value: {explanation.base_values[0]:.4f}")

## 3. Local Explanations (Single Instance)

In [ ]:
# Explain a single prediction
sample_idx = 0
sample = X_test.iloc[sample_idx:sample_idx+1]

# Get prediction
prediction = model.predict(sample)[0]
prediction_proba = model.predict_proba(sample)[0]

print(f"Sample {sample_idx}:")
print(f"  Prediction: {prediction} ({'Benign' if prediction == 1 else 'Malignant'})")
print(f"  Probability: {prediction_proba[1]:.4f}")

In [ ]:
# Waterfall plot
shap.plots.waterfall(explanation[0], max_display=15, show=True)

In [ ]:
# Force plot for single instance
shap.initjs()
shap.force_plot(tree_explainer.expected_value, tree_shap_values[0], X_test.iloc[0])

In [ ]:
# Understanding the contribution
sample_shap = tree_shap_values[0]
feature_contributions = list(zip(X_test.columns, sample_shap, X_test.iloc[0].values))
feature_contributions.sort(key=lambda x: abs(x[1]), reverse=True)

print("\nTop 10 Feature Contributions:")
print("-" * 60)
print(f"{'Feature':<25} {'SHAP Value':>12} {'Feature Value':>15}")
print("-" * 60)
for feat, shap_val, feat_val in feature_contributions[:10]:
    direction = "↑" if shap_val > 0 else "↓"
    print(f"{feat:<25} {shap_val:>12.4f} {direction} {feat_val:>12.4f}")

## 4. Global Explanations (Model-wide)

In [ ]:
# Global feature importance
shap.summary_plot(tree_shap_values, X_test[:50], plot_type="bar", show=True)

In [ ]:
# Summary plot with feature values
shap.summary_plot(tree_shap_values, X_test[:50], show=True)

In [ ]:
# Calculate mean absolute SHAP values
mean_abs_shap = np.abs(tree_shap_values).mean(axis=0)
feature_importance = pd.DataFrame({
    'feature': X_test.columns,
    'importance': mean_abs_shap
}).sort_values('importance', ascending=False)

print("Global Feature Importance (SHAP):")
print(feature_importance.head(15).to_string(index=False))

## 5. Dependence Plots

In [ ]:
# Dependence plot for top feature
top_feature = feature_importance.iloc[0]['feature']
print(f"Dependence plot for: {top_feature}")

shap.dependence_plot(top_feature, tree_shap_values, X_test[:50], show=True)

In [ ]:
# Multiple dependence plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for i, ax in enumerate(axes.flat):
    if i < 4:
        feat = feature_importance.iloc[i]['feature']
        feat_idx = list(X_test.columns).index(feat)
        ax.scatter(
            X_test.iloc[:50, feat_idx],
            tree_shap_values[:50, feat_idx],
            c=y_test[:50],
            cmap='coolwarm',
            alpha=0.6
        )
        ax.set_xlabel(feat)
        ax.set_ylabel('SHAP value')
        ax.set_title(f'{feat}')
        ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)

plt.tight_layout()
plt.show()

## 6. SHAP Interaction Values

In [ ]:
# Calculate interaction values (computationally intensive)
# Only use small sample
interaction_values = tree_explainer.shap_interaction_values(X_test[:20])

print(f"Interaction values shape: {interaction_values.shape}")

In [ ]:
# Interaction summary
# Main effects on diagonal, interactions off-diagonal
mean_interactions = np.abs(interaction_values).mean(axis=0)

# Find top interactions
n_features = len(X_test.columns)
interactions = []

for i in range(n_features):
    for j in range(i+1, n_features):
        interactions.append({
            'feature_1': X_test.columns[i],
            'feature_2': X_test.columns[j],
            'interaction': mean_interactions[i, j]
        })

interactions_df = pd.DataFrame(interactions).sort_values('interaction', ascending=False)
print("Top Feature Interactions:")
print(interactions_df.head(10).to_string(index=False))

## 7. Comparing Predictions

In [ ]:
# Compare explanations for different predictions
# Find samples with different predictions
predictions = model.predict(X_test[:50])
benign_idx = np.where(predictions == 1)[0][0]
malignant_idx = np.where(predictions == 0)[0][0]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Benign prediction
ax1 = axes[0]
benign_shap = tree_shap_values[benign_idx]
sorted_idx = np.argsort(np.abs(benign_shap))[::-1][:10]
ax1.barh(range(10), benign_shap[sorted_idx][::-1], 
         color=['green' if v > 0 else 'red' for v in benign_shap[sorted_idx][::-1]])
ax1.set_yticks(range(10))
ax1.set_yticklabels([X_test.columns[i] for i in sorted_idx[::-1]])
ax1.set_title(f'Benign Prediction (Sample {benign_idx})')
ax1.set_xlabel('SHAP Value')
ax1.axvline(x=0, color='black', linewidth=0.5)

# Malignant prediction
ax2 = axes[1]
malignant_shap = tree_shap_values[malignant_idx]
sorted_idx = np.argsort(np.abs(malignant_shap))[::-1][:10]
ax2.barh(range(10), malignant_shap[sorted_idx][::-1],
         color=['green' if v > 0 else 'red' for v in malignant_shap[sorted_idx][::-1]])
ax2.set_yticks(range(10))
ax2.set_yticklabels([X_test.columns[i] for i in sorted_idx[::-1]])
ax2.set_title(f'Malignant Prediction (Sample {malignant_idx})')
ax2.set_xlabel('SHAP Value')
ax2.axvline(x=0, color='black', linewidth=0.5)

plt.tight_layout()
plt.show()

## Summary

### Key Takeaways:

1. **SHAP values** explain individual predictions by showing how each feature contributes
2. **Positive SHAP values** push the prediction higher, negative values push it lower
3. **Sum of SHAP values** equals the difference between prediction and base value
4. **Global importance** can be computed by averaging absolute SHAP values
5. **Dependence plots** show how feature values relate to their SHAP contributions
6. **Interaction values** reveal how features work together

### Best Practices:
- Use **TreeExplainer** for tree-based models (fast)
- Use **KernelExplainer** for any model (slow but model-agnostic)
- Start with **summary plots** for overview
- Use **waterfall plots** for individual explanations
- Check **dependence plots** for non-linear relationships